In [32]:
import numpy as np
from PIL import Image
from colorsys import hsv_to_rgb

def gen_grad(width, c1, c2): 
    t = np.linspace(0, 1, width)
    return np.array([c1[idx] + t * (c2[idx] - c1[idx]) for idx in range(3)]).T

def from_hsv(img):
    w, h, _ = img.shape
    new_img = np.zeros_like(img)
    for i in range(w):
        for j in range(h):
            new_img[i, j, :] = 255 * np.array(hsv_to_rgb(*img[i, j, :]))
    return new_img

def array_to_image(color_array, width, height):
    color_array = from_hsv(color_array)
    img_array = np.array(color_array, dtype=np.uint8)
    img_array = img_array.reshape((height, width, 3))
    img = Image.fromarray(img_array)
    return img

# Параметры
start_width = 100
start_height = 60
depth = 5
c1, c2 = (170 / 360, 0.7, 0.7), (230 / 360, 0.7, 0.7)

# Рассчитываем размеры изображения
height = start_height * (2 ** depth - 1)
width = start_width * (2 ** depth)

# Создаем изображение
img = np.zeros((height, width, 3))

for step in range(1, depth + 1):
    # Количество полос на текущем уровне
    num_strips = 2 ** (step - 1)
    # Ширина каждой полосы уменьшается с каждым уровнем
    strip_width = width // num_strips
    # Высота текущего уровня
    level_height = start_height * (2 ** (step - 1))
    
    # Начальная позиция по Y для текущего уровня
    y_start = sum(start_height * (2 ** (s - 1)) for s in range(1, step))
    
    for strip_idx in range(num_strips):
        # Позиция по X для текущей полосы
        x_start = strip_idx * strip_width
        x_end = (strip_idx + 1) * strip_width
        
        # Создаем градиент для текущей полосы
        grad = gen_grad(strip_width, c1, c2)
        
        # ЧЕРЕДОВАНИЕ: нечетные полосы - отраженные градиенты
        if strip_idx % 2 == 1:
            grad = np.flip(grad, axis=0)
        
        # Заполняем текущую полосу для всех строк уровня
        for y in range(y_start, y_start + level_height):
            img[y, x_start:x_end, :] = grad

# Создаем и показываем изображение
image = array_to_image(img, width, height)
image.show()
image.save('test.png')